# 01 - Basic SGD Implementation

This notebook demonstrates the fundamental concepts of Stochastic Gradient Descent (SGD) and its dynamics on a simple loss landscape.

**Converted from:** `sgd_example.nb` (Mathematica)

## Contents:
1. Data generation with noise
2. Loss landscape visualization  
3. SGD trajectory tracking
4. Parameter evolution over time

## Background

SGD is a fundamental optimization algorithm in machine learning. In this notebook, we'll explore how SGD navigates a loss landscape when fitting a nonlinear function to noisy data.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add utils to path
sys.path.insert(0, str(Path.cwd() / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig, sgd_trajectory
from loss_functions import (
    smooth_nonlinear_loss, 
    smooth_nonlinear_gradient,
    generate_noisy_data,
    true_function,
    SmoothNonlinearLoss
)
from visualization import (
    plot_loss_landscape,
    plot_loss_landscape_3d,
    plot_parameter_evolution,
    plot_trajectories
)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

## 1. Generate Noisy Data

We'll generate data from a smooth nonlinear function with added Gaussian noise:

$$f(x; p) = x \left(1 + \frac{p}{1 + e^{-x}}\right)$$

This represents a realistic scenario where we have noisy observations from an unknown function.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Data generation parameters
x_range = (-3, 3)  # Range of x values
n_points = 20       # Number of distinct x points
n_samples_per_point = 5  # Multiple noisy samples per x
noise_std = 0.5     # Standard deviation of noise
p_true = 1.0        # True parameter value

# Generate noisy data
x_data, y_data = generate_noisy_data(
    x_range=x_range,
    n_points=n_points,
    n_samples_per_point=n_samples_per_point,
    noise_std=noise_std,
    p=p_true,
    random_state=42
)

print(f"Generated {len(x_data)} data points")
print(f"X range: [{x_data.min():.2f}, {x_data.max():.2f}]")
print(f"Y range: [{y_data.min():.2f}, {y_data.max():.2f}]")

In [ ]:
# Visualize the data and true function
fig, ax = plt.subplots(figsize=(12, 7))

# Plot noisy data points
ax.scatter(x_data, y_data, alpha=0.5, s=50, label='Noisy observations', color='blue')

# Plot true function
x_smooth = np.linspace(x_range[0], x_range[1], 200)
y_true = true_function(x_smooth, p=p_true)
ax.plot(x_smooth, y_true, 'r-', linewidth=2.5, label='True function', alpha=0.8)

ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('y', fontsize=13)
ax.set_title('Noisy Data from Nonlinear Function', fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Loss Landscape Visualization

The loss function we're optimizing is:

$$L(a, b) = \frac{1}{N} \sum_{i=1}^N \left(y_i - x_i\left(1 + \frac{b}{1 + e^{-ax_i}}\right)\right)^2$$

We'll visualize this loss landscape in 2D (contour plot) and 3D.

In [ ]:
# Define parameter ranges for visualization
param_range = ((-2, 4), (-2, 4))  # ((a_min, a_max), (b_min, b_max))

# Create loss function object
loss_obj = SmoothNonlinearLoss(p=p_true)

# Plot 2D loss landscape
fig, ax = plt.subplots(figsize=(12, 10))
plot_loss_landscape(
    loss_fn=loss_obj,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=100,
    contour_levels=40,
    ax=ax,
    title='Loss Landscape - Contour Plot'
)
plt.tight_layout()
plt.show()

print("The loss landscape shows multiple local minima and complex structure!")

In [ ]:
# Plot 3D loss landscape
fig, ax = plot_loss_landscape_3d(
    loss_fn=loss_obj,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=60,
    title='3D Loss Landscape'
)
plt.tight_layout()
plt.show()

## 3. SGD Trajectory Tracking

Now we'll run SGD from different initial conditions and observe how it navigates the loss landscape.

The SGD update rule is:

$$\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t; \text{minibatch})$$

where $\eta$ is the learning rate.

In [ ]:
# Configure SGD
sgd_config = SGDConfig(
    learning_rate=0.05,
    batch_size=10,
    n_iterations=2000,
    random_state=42
)

# Initial parameters (starting from a suboptimal point)
initial_params = np.array([0.5, 0.5])

# Create simulator and run trajectory
simulator = SGDSimulator(sgd_config)
trajectory, iterations = simulator.run_trajectory(
    initial_params=initial_params,
    gradient_fn=loss_obj.gradient,
    x_data=x_data,
    y_data=y_data,
    save_every=10
)

print(f"Initial parameters: {initial_params}")
print(f"Final parameters: {trajectory[-1]}")
print(f"Initial loss: {loss_obj(initial_params, x_data, y_data):.4f}")
print(f"Final loss: {loss_obj(trajectory[-1], x_data, y_data):.4f}")

In [ ]:
# Plot loss landscape with trajectory overlay
fig, ax = plt.subplots(figsize=(14, 11))
plot_loss_landscape(
    loss_fn=loss_obj,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=100,
    trajectory=trajectory,
    contour_levels=40,
    ax=ax,
    title='SGD Trajectory on Loss Landscape'
)
plt.tight_layout()
plt.show()

## 4. Parameter Evolution Over Time

Let's examine how each parameter evolves during the optimization.

In [ ]:
# Plot parameter evolution
fig, ax = plt.subplots(figsize=(14, 7))
plot_parameter_evolution(
    iterations=iterations,
    trajectory=trajectory,
    param_names=['Parameter a', 'Parameter b'],
    ax=ax,
    title='Parameter Evolution During SGD'
)
plt.tight_layout()
plt.show()

In [ ]:
# Plot loss evolution
losses = [loss_obj(params, x_data, y_data) for params in trajectory]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(iterations, losses, 'b-', linewidth=2, alpha=0.7)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Evolution During SGD', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss decreased by {((losses[0] - losses[-1])/losses[0] * 100):.1f}%")

## 5. Multiple Trajectories from Different Initializations

Let's run SGD from multiple random starting points to see how initial conditions affect the final solution.

In [ ]:
# Run multiple trajectories
np.random.seed(123)
n_trajectories = 5
trajectories = []

for i in range(n_trajectories):
    # Random initialization
    init_params = np.random.uniform(-1, 3, size=2)
    
    traj, iters = simulator.run_trajectory(
        initial_params=init_params,
        gradient_fn=loss_obj.gradient,
        x_data=x_data,
        y_data=y_data,
        save_every=10
    )
    
    trajectories.append(traj)
    print(f"Trajectory {i+1}: Start {init_params} -> End {traj[-1]}")

In [ ]:
# Plot all trajectories on loss landscape
fig, ax = plt.subplots(figsize=(14, 11))

# Create grid for loss landscape
param_range_wide = ((-1.5, 3.5), (-1.5, 3.5))
(a_min, a_max), (b_min, b_max) = param_range_wide
n_points = 100
a_vals = np.linspace(a_min, a_max, n_points)
b_vals = np.linspace(b_min, b_max, n_points)
A, B = np.meshgrid(a_vals, b_vals)

# Compute loss
Z = np.zeros_like(A)
for i in range(n_points):
    for j in range(n_points):
        params = np.array([A[i, j], B[i, j]])
        Z[i, j] = loss_obj(params, x_data, y_data)

# Plot contours
contourf = ax.contourf(A, B, Z, levels=30, cmap='viridis', alpha=0.3)
ax.contour(A, B, Z, levels=30, cmap='viridis', alpha=0.6)
plt.colorbar(contourf, ax=ax, label='Loss')

# Plot all trajectories
colors = plt.cm.tab10(np.linspace(0, 1, n_trajectories))
for i, traj in enumerate(trajectories):
    ax.plot(traj[:, 0], traj[:, 1], '-', linewidth=2, 
           color=colors[i], alpha=0.8, label=f'Traj {i+1}')
    ax.plot(traj[0, 0], traj[0, 1], 'o', markersize=10, 
           color=colors[i], markeredgecolor='black', markeredgewidth=2)
    ax.plot(traj[-1, 0], traj[-1, 1], 's', markersize=10,
           color=colors[i], markeredgecolor='black', markeredgewidth=2)

ax.set_xlabel('Parameter a', fontsize=12)
ax.set_ylabel('Parameter b', fontsize=12)
ax.set_title('Multiple SGD Trajectories from Different Initializations', fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we demonstrated:

1. **Data Generation**: Created noisy observations from a smooth nonlinear function
2. **Loss Landscape**: Visualized the complex 2D loss surface with multiple minima
3. **SGD Dynamics**: Tracked how SGD navigates the loss landscape
4. **Parameter Evolution**: Observed how parameters change over iterations
5. **Multiple Initializations**: Showed that different starting points can lead to different local minima

**Key Observations:**
- The loss landscape is non-convex with multiple local minima
- SGD successfully reduces loss but may converge to different solutions
- The stochastic nature of SGD causes the trajectory to be noisy
- Learning rate and batch size significantly affect the optimization path

**Next Steps:**
- In the next notebook, we'll explore the connection between SGD and Langevin dynamics
- We'll study the fluctuation-dissipation relation and equilibrium conditions